In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

# Load dataset
data = load_diabetes(as_frame=True)

df = data.frame

print(df.head())
print("Dataset shape:", df.shape)
print(df.columns)

        age       sex       bmi        bp        s1        s2        s3  \
0  0.038076  0.050680  0.061696  0.021872 -0.044223 -0.034821 -0.043401   
1 -0.001882 -0.044642 -0.051474 -0.026328 -0.008449 -0.019163  0.074412   
2  0.085299  0.050680  0.044451 -0.005670 -0.045599 -0.034194 -0.032356   
3 -0.089063 -0.044642 -0.011595 -0.036656  0.012191  0.024991 -0.036038   
4  0.005383 -0.044642 -0.036385  0.021872  0.003935  0.015596  0.008142   

         s4        s5        s6  target  
0 -0.002592  0.019907 -0.017646   151.0  
1 -0.039493 -0.068332 -0.092204    75.0  
2 -0.002592  0.002861 -0.025930   141.0  
3  0.034309  0.022688 -0.009362   206.0  
4 -0.002592 -0.031988 -0.046641   135.0  
Dataset shape: (442, 11)
Index(['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6',
       'target'],
      dtype='object')


In [3]:
x=df[['bmi','age']].values
y=df['target'].values

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
x_train.shape

(353, 2)

In [5]:
def poly(x,degree):
    x=np.asarray(x,dtype=float)
    x=x[:,1]

    columns=[x**power for power in range(1,degree+1)]
    return np.column_stack(columns)

x_poly=poly(x_train,degree=2)
x_poly.shape

(353, 2)

In [7]:
class PolynomialRegression:
    def __init__(self, degree=2, lr=0.01, epochs=10000):
        self.degree = degree
        self.lr = lr
        self.epochs = epochs

    def fit(self, X, y):
        X_poly = poly(X, self.degree)
        y = np.asarray(y).ravel()

        n_samples, n_features = X_poly.shape

        self.weights = np.zeros(n_features)
        self.bias = 0.0
        self.loss_history = []

        for epoch in range(self.epochs):
            # Forward pass
            y_pred = X_poly @ self.weights + self.bias

            # Errors
            error = y_pred - y

            # Gradients
            dw = (2 / n_samples) * (X_poly.T @ error)
            db = 2 * np.mean(error)

            # Update parameters
            self.weights -= self.lr * dw
            self.bias -= self.lr * db

            # Track loss
            if epoch % 100 == 0:
                loss = np.mean(error ** 2)
                self.loss_history.append(loss)

        return self

    def predict(self, X):
        X_poly = poly(X, self.degree)
        return X_poly @ self.weights + self.bias

In [8]:
model = PolynomialRegression(
    degree=2,
    lr=0.01,
    epochs=10000
)

model.fit(x_train, y_train)

y_pred = model.predict(x_test)

print("Weights:", model.weights)
print("Bias:", model.bias)

Weights: [115.45261146  -1.32935512]
Bias: 153.57358402264953


In [9]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mse = mean_squared_error(y_test, y_pred)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mse)
print("RMSE:", np.sqrt(mse))
print("R²:", r2_score(y_test, y_pred))

MAE: 63.250828112122115
MSE: 5255.606478782784
RMSE: 72.49556178679343
R²: 0.008030209732239224


In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load real-world dataset
data = load_diabetes(as_frame=True)

X = data.data[["bmi"]]  # Use BMI as a single feature for visualization
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (353, 1)
Testing shape: (89, 1)


In [11]:
model = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

In [12]:
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R²:", r2_score(y_test, y_pred))

MAE: 52.38391176015265
MSE: 4085.0254808716336
RMSE: 63.91420406194255
R²: 0.22897159712056647
